# CCA significance — Wilks' lambda (parametric)

For each metric (`age_effects`, `similarity_strength`) the cell-density block `X`
(N parcels × 23 cell types) is related to a single cortical metric by CCA. With one
metric on the Y side, `min(p, q) = 1`: there is a single canonical correlation equal
to the multiple-*R* of the metric on `X`, and Wilks' lambda reduces to

$$\Lambda = 1 - R^2,$$


In [1]:
import numpy as np
import pandas as pd
import nibabel as nib
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from pathlib import Path

In [ ]:
# ---- paths (edit to your machine) ----
ROOT = Path.cwd().parents[0] 

MAC_DIR  = ROOT / 'data' / 'gifti' / 'CIVET_macaque-alpha-0.2'
LH_LABEL = MAC_DIR / 'D99_atlas_rsl_sym_left.label.gii'
CELL_CSV = ROOT / 'data' / 'd99_cell_abundance.csv'
AGE_CSV  = ROOT / 'Mixed_Effects_Models' / 'regionwise_age_effects_MixedLM.csv'
SIM_CSV  = ROOT / 'MIND_Network' / 'total_similarity_strength.csv'

ID_COL  = 'D99'
EXCLUDE = (70,)
ALPHA   = 0.05
METRICS = ['age_effects', 'similarity_strength']

In [3]:
# ---- feature block: metrics indexed by D99 region ----
feat_df = (pd.read_csv(AGE_CSV)
             .set_index('region')[['age_t']]
             .rename(columns={'age_t': 'age_effects'}))
feat_df['similarity_strength'] = (pd.read_csv(SIM_CSV)
                                    .set_index('region')['total_similarity_strength'])

# ---- cell-type block ----
cell_df = pd.read_csv(CELL_CSV).set_index(ID_COL)

In [4]:
# ---- parcel order: kept D99 regions (drop background and EXCLUDE) ----
lab = nib.load(LH_LABEL).darrays[0].data.astype(int)
region_order = list(np.unique(lab[(lab > 0) & ~np.isin(lab, list(EXCLUDE))]))
N = len(region_order)

def align(df, name):
    miss = [r for r in region_order if r not in df.index]
    if miss:
        raise ValueError(f'{name}: {len(miss)} parcels missing from index (e.g. {miss[:5]}).')
    return df.reindex(region_order)

cell_block = align(cell_df, 'cells')
feat_block = align(feat_df, 'features')
cell_names = list(cell_block.columns)

X = cell_block.to_numpy(float)                       # N x 23
assert np.isfinite(X).all(), 'NaNs in cell block.'
for m in METRICS:
    assert m in feat_block.columns, f'metric {m!r} missing.'
    assert np.isfinite(feat_block[m].to_numpy(float)).all(), f'NaNs in metric {m}.'

print(f'aligned X: {X.shape}; cell types: {len(cell_names)}; metrics: {METRICS}')

aligned X: (140, 23); cell types: 23; metrics: ['age_effects', 'similarity_strength']


In [5]:
# ---- CCA, Wilks' lambda (q=1: Wilks LR test == OLS omnibus F) ----
rank_X = np.linalg.matrix_rank(X - X.mean(0))
if rank_X < X.shape[1]:
    print(f'WARNING: cell block rank {rank_X} < {X.shape[1]} columns (compositional data); '
          f'the nominal numerator df overstates and the parametric p is optimistic.\n')

rows = []
for m in METRICS:
    y = feat_block[m].to_numpy(float)
    model = sm.OLS(y, sm.add_constant(X)).fit()
    R2 = float(model.rsquared)
    rows.append(dict(metric=m,
                     can_corr=float(np.sqrt(max(R2, 0.0))),
                     R2=R2,
                     wilks_lambda=1.0 - R2,
                     F=float(model.fvalue),
                     num_df=float(model.df_model),
                     den_df=float(model.df_resid),
                     p_param=float(model.f_pvalue)))

wilks = pd.DataFrame(rows)
wilks['p_param_fdr'] = multipletests(wilks['p_param'], method='fdr_bh')[1]
with pd.option_context('display.float_format', lambda v: f'{v:.4g}'):
    print(wilks.to_string(index=False))

             metric  can_corr     R2  wilks_lambda     F  num_df  den_df   p_param  p_param_fdr
        age_effects    0.5886 0.3465        0.6535 2.674      23     116 0.0003053    0.0006107
similarity_strength    0.5265 0.2772        0.7228 1.934      23     116    0.0121       0.0121
